# 🛠️ Data Engineering Technical Audit

Notebook ini berfokus pada **Data Profiling** dan **Pipeline Verification**. Tujuannya adalah memastikan integritas data dari tahap Raw hingga Star Schema.

---

In [1]:
import duckdb
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

DB_PATH = "../data/final/tlc.duckdb"
con = duckdb.connect(DB_PATH, read_only=True)

sns.set_theme(style="white")
plt.rcParams['figure.figsize'] = (12, 6)

print(f"✅ Audit Engine Ready: {DB_PATH}")

✅ Audit Engine Ready: ../data/final/tlc.duckdb


## 1. Pipeline Volume Tracking
Memverifikasi jumlah baris yang diproses di setiap tahap ELT.

In [2]:
volume_query = """
SELECT '1. Raw (tlc_raw)' as Stage, COUNT(*) as Row_Count FROM tlc_raw
UNION ALL
SELECT '2. Cleaned (tlc_cleaned)' as Stage, COUNT(*) as Row_Count FROM tlc_cleaned
UNION ALL
SELECT '3. Fact (fact_trips)' as Stage, COUNT(*) as Row_Count FROM fact_trips
"""
df_vol = con.execute(volume_query).df()
df_vol['Loss_Rate (%)'] = (1 - (df_vol['Row_Count'] / df_vol['Row_Count'].iloc[0])) * 100

print("📊 Data Volume Pipeline:")
display(df_vol)

plt.figure(figsize=(10, 5))
sns.barplot(data=df_vol, x='Stage', y='Row_Count', palette='Blues_d')
plt.title('Data Volume Retention per Pipeline Stage')
plt.show()

🔍 Top NULL Columns in fact_trips:


,column,null_rows,null_rate (%)
26,dropoff_latitude,136212,0.7520
25,dropoff_longitude,136212,0.7520
23,pickup_longitude,50406,0.2783
24,pickup_latitude,50406,0.2783


## 2. NULL Rate Analysis (Column Integrity)
Menganalisis kolom mana yang paling banyak memiliki data kosong.

In [3]:
def get_null_report(table_name):
    cols = con.execute(f"PRAGMA table_info({table_name})").df()['name'].tolist()
    null_counts = []
    total_rows = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    
    for col in cols:
        count = con.execute(f"SELECT COUNT(*) FROM {table_name} WHERE \"{col}\" IS NULL").fetchone()[0]
        null_counts.append({'column': col, 'null_rows': count, 'null_rate (%)': round((count/total_rows)*100, 4)})
    
    return pd.DataFrame(null_counts).sort_values('null_rows', ascending=False)

print("🔍 Top NULL Columns in fact_trips:")
fact_nulls = get_null_report('fact_trips')
display(fact_nulls[fact_nulls['null_rows'] > 0])

🔍 Top NULL Columns in fact_trips:


,column,null_rows,null_rate (%)
26,dropoff_latitude,136212,0.7520
25,dropoff_longitude,136212,0.7520
23,pickup_longitude,50406,0.2783
24,pickup_latitude,50406,0.2783


## 3. Referential Integrity Check (Star Schema)
Memastikan semua Foreign Key di Tabel Fakta memiliki pasangan di Tabel Dimensi.

In [4]:
integrity_checks = {
    "Orphaned Locations": "SELECT COUNT(*) FROM fact_trips f LEFT JOIN dim_location l ON f.pickup_location_key = l.location_key WHERE l.location_key IS NULL",
    "Missing Weather Links": "SELECT COUNT(*) FROM fact_trips WHERE weather_key = -1",
    "Time Dimension Coverage": "SELECT COUNT(*) FROM fact_trips f LEFT JOIN dim_time t ON f.time_key = t.time_key WHERE t.time_key IS NULL"
}

print("✅ Star Schema Integrity Results:")
for label, sql in integrity_checks.items():
    count = con.execute(sql).fetchone()[0]
    status = "FAIL ❌" if count > 0 else "PASS ✅"
    print(f"{status} {label}: {count} rows")

✅ Star Schema Integrity Results:
PASS ✅ Orphaned Locations: 0 rows
PASS ✅ Missing Weather Links: 0 rows
PASS ✅ Time Dimension Coverage: 0 rows


## 4. Anomaly Filtering Verification
Memeriksa apakah aturan pembersihan data (Cleaning Rules) benar-benar diterapkan.

In [5]:
rules_check = {
    "Negative Fares": "SELECT COUNT(*) FROM fact_trips WHERE fare_amount < 0",
    "Zero Passengers": "SELECT COUNT(*) FROM fact_trips WHERE passenger_count <= 0",
    "Invalid Duration (>5hr)": "SELECT COUNT(*) FROM fact_trips WHERE trip_duration_min > 300",
    "Future Dates (> 2025)": "SELECT COUNT(*) FROM fact_trips WHERE pickup_datetime > '2025-12-31'"
}

print("🛡️ Cleaning Rule Compliance:")
for label, sql in rules_check.items():
    count = con.execute(sql).fetchone()[0]
    status = "CLEANED ✅" if count == 0 else "ISSUE FOUND ⚠️"
    print(f"{status} {label}: {count} anomalies found in fact table")

🛡️ Cleaning Rule Compliance:
CLEANED ✅ Negative Fares: 0 anomalies found in fact table
CLEANED ✅ Zero Passengers: 0 anomalies found in fact table
CLEANED ✅ Invalid Duration (>5hr): 0 anomalies found in fact table
CLEANED ✅ Future Dates (> 2025): 0 anomalies found in fact table


## 5. Storage Metrics (DuckDB Efficiency)
Melihat ukuran tabel dan efisiensi penyimpanan.

In [6]:
print("💾 Storage Profiling:")
tables_df = con.execute("SHOW TABLES").df()
for table in tables_df['name']:
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"- Table '{table}': {count:,} records")

💾 Storage Profiling:
- Table 'dim_location': 263 records
- Table 'dim_time': 181 records
- Table 'dim_weather': 181 records
- Table 'fact_trips': 18,112,530 records
- Table 'taxi_zone_lookup': 265 records
- Table 'tlc_cleaned': 18,112,530 records
- Table 'tlc_loaded_files': 6 records
- Table 'tlc_raw': 24,083,384 records
- Table 'weather_raw': 181 records
- Table 'weather_transformed': 181 records


In [7]:
con.close()
print("--- Technical Audit Finished ---")

--- Technical Audit Finished ---
